# Model Iteration Notebook 

In this notebook, we will focus on training and evaluating multiple machine learning models to classify emotions in transcribed television show data. 

The different secitons in this notebook will help demonstrate how different models perform on the same dataset and will provide insights into model selection and optimization. 

For every type of model and/or iteration that we did, we tracked the details in the model iteration file that you can find attached to the final assignment on Brightspace or here [Model Iteration File](link-here). In order to improve our models, we used (a selection of) traditional NLP features that we extracted in the ‘NLP Features’ task [NLP Features.ipynb notebook](NLP Features.ipynb). 



In [1]:
import numpy as np
import pandas as pd 
from datasets import DatasetDict, load_dataset

c:\Users\rzvn1\.conda\envs\nlp_env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import torch 

print("Torch version:",torch.__version__)

print("Is CUDA enabled?",torch.cuda.is_available())

Torch version: 2.5.1
Is CUDA enabled? True


## Loading the datasets

### Dataset 1 - Sentiment and Emotion Analysis Dataset

The dataset can be found at [https://www.kaggle.com/datasets/kushagra3204/sentiment-and-emotion-analysis-dataset?resource=download](https://www.kaggle.com/datasets/kushagra3204/sentiment-and-emotion-analysis-dataset?resource=download)

The dataset contains over 422,000 sentences, labeled with six distinct emotions:

- Joy: 143,067 samples
- Sadness: 121,187 samples
- Anger: 59,317 samples
- Fear: 49,649 samples
- Love: 34,554 samples
- Surprise: 14,972 samples


In [3]:
# https://www.kaggle.com/datasets/kushagra3204/sentiment-and-emotion-analysis-dataset?resource=download
# Sentiment and Emotion Analysis Dataset

df1 = pd.read_csv(r"..\Data\CSV\sentiment_data\combined_emotion.csv")
df1['emotion'].value_counts()

emotion
joy        143067
sad        121187
anger       59317
fear        49649
love        34554
suprise     14972
Name: count, dtype: int64

### Dataset 2 - roskoN/dailydialog

The dataset can be found at [https://huggingface.co/datasets/roskoN/dailydialog](https://huggingface.co/datasets/roskoN/dailydialog)

The dataset contains 102,979 sentences, labeled with seven distinct emotions, including neutral:

- Neutral: 85572
- Happiness: 12885
- Surprise: 1823
- Hadness: 1150
- Anger: 1022
- Disgust: 353
- Fear: 174


In [4]:
ds = load_dataset("roskoN/dailydialog")

## Preprocessing

### Dataset 1 

1. Emotion Harmonization: The emotion `"love"` was removed because it did not align with the six primary emotions used in the final schema.

2. Name Standardization: The emotion names `"joy"` and `"sad"` were renamed to `"happiness"` and `"sadness"`, respectively, to match the target naming convention.  

3. Typo Correction: A typo was corrected by changing the misspelled emotion `"suprise"` to the correct spelling, `"surprise"`, resolving mapping issues. 

4. Column Renaming: The column names `sentence` and `emotion` were capitalized to `Sentence` and `Emotion` to match the consistent naming used for Dataset 2.

In [5]:
values_to_remove = ['love']

mask_to_keep = ~df1['emotion'].isin(values_to_remove) # ~ Means reverse condition
df1 = df1[mask_to_keep]

In [6]:
df1.replace('joy', 'happiness', inplace=True) # Replace joy with happiness to match the naming convention
df1.replace('sad', 'sadness', inplace=True) # Replace sad with sadness to match the naming convention
df1.replace('suprise', 'surprise', inplace=True) # Fix typo suprise --> surprise

In [7]:
df1.rename(columns={
    'sentence': 'Sentence',
    'emotion': 'Emotion'
}, inplace=True)

### Dataset 2 

Here's a slightly rephrased and condensed explanation of the steps taken:

To prepare the data:

1. Emotion Mapping: The specific numerical IDs were obtained by accessing the original research paper linked on Hugging Face. The paper can be found at [https://aclanthology.org/I17-1099/](https://aclanthology.org/I17-1099/). I then downloaded the provided zip file (`I17-1099.Datasets.zip`), and inside it, the file `readme.txt` contained the ID-to-emotion mapping we used.

2. Reverse Map Creation: A secondary, inverted dictionary was created to map the emotion names back to their corresponding IDs. This was done to ensure consistency and standardize the emotion IDs across multiple datasets (e.g., mapping emotion names in Dataset 1 to match the ID scheme of this dataset).

3. Dataframe Conversion & Unpacking: The Hugging Face DatasetDict object was converted into a pandas DataFrame. During this process, rows originally containing a list of multiple sentences and their corresponding emotions were unpacked to create a single row for every individual sentence-emotion pair.

In [8]:
# Emotion Map obtained from https://aclanthology.org/I17-1099/ --> Download I17-1099.Datasets.zip --> readme.txt 
emotion_map = {0: "neutral", 
              1: "anger", 
              2: "disgust", 
              3: "fear", 
              4: "happiness", 
              5: "sadness", 
              6: "surprise"} 

# Rever Emotion Map to match the IDs of dataset 1 with the ones of dataset 2 
reverse_map  = {v: k for k, v in emotion_map.items()}

In [9]:
def make_dataset_into_df(dataset: DatasetDict) -> pd.DataFrame:
    """
    Flattens a Hugging Face DatasetDict into a single pandas DataFrame
    where each row is a sentence-emotion pair.
    """
    all_sentences = []
    all_emotions = []
    all_splits = []

    # Iterate through each split ('train', 'validation', 'test')
    for split_name, ds_split in dataset.items():
        # Iterate through each dialogue/example in the split
        for example in ds_split:
            # 'utterances' and 'emotions' are lists
            for sentence, emotion in zip(example['utterances'], example['emotions']):
                all_sentences.append(sentence)
                all_emotions.append(emotion)
                all_splits.append(split_name) # Keep track of which split it came from

    # Create the DataFrame from the collected lists
    df = pd.DataFrame({
        'Sentence': all_sentences,
        'Emotion_ID': all_emotions,
        'Split': all_splits  
    })

    return df

In [10]:
df2 = make_dataset_into_df(ds)        

In [11]:
df2['Emotion'] = df2['Emotion_ID'].map(emotion_map)

## Merging the Datasets

In [12]:
df_concat = pd.concat([df1, df2], ignore_index=True)

In [13]:
df_concat['Emotion_ID'] = df_concat['Emotion_ID'].fillna(df_concat['Emotion'].map(reverse_map))
df_concat['Emotion_ID'].astype(int)
df_concat['Emotion'].value_counts()

Emotion
happiness    155952
sadness      122337
neutral       85572
anger         60339
fear          49823
surprise      16795
disgust         353
Name: count, dtype: int64